<a href="https://colab.research.google.com/github/shahwaiz-9/Deep-Learning/blob/main/Ai_Text_Detector_Using_Hugging_Face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Install Required Libraries

First, we need to install `python-docx` to read `.docx` files and `transformers` for accessing Hugging Face models.

In [1]:
pip install python-docx transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.8 MB/s eta 0:00:00


### Step 2: Read Your Uploaded Document

Since you've uploaded your document to `/content/keratoconus_final_draft.docx`, we'll directly read its content.

In [ ]:
# This cell previously created a dummy document. Since you've uploaded your own,
# we'll no longer create a dummy one. If you want to use a specific document,
# make sure its path is correctly set in the next cell.

### Step 3: Read Text from the `.docx` File

Now, let's read the content from the `info.docx` file we just created (or your own file if you're running this locally).

In [6]:
from docx import Document

def read_docx(file_path):
    doc = Document(file_path)
    full_text = []
    for para in doc.paragraphs:
        full_text.append(para.text)
    return '\n'.join(full_text)

# Read the content of the user's uploaded document
document_path = '/content/keratoconus_final_draft.docx'
file_content = read_docx(document_path)
print("--- Document Content (first 500 characters) ---")
print(file_content[:40000])

--- Document Content (first 500 characters) ---
KC-ViT: A Multimodal Vision Transformer for Keratoconus Detection on the CornOrb Dataset
Daniyal Ahmed, Shahwaiz Ali, Zia ur Rehman and Quratul Ain


Abstract—  Keratoconus is a progressive corneal disorder for which early, automated detection remains an open challenge due to the scarcity of large, publicly available, multimodal datasets. This study proposes and evaluates a multimodal deep learning architecture, KeratoconusMultimodalNet, on CornOrb, a recently released dataset of 1,454 eyes with four co-registered corneal maps (Anterior, Axial, Pachymetry, Posterior) and structured clinical annotations. The architecture fuses a composite image representation, formed by spatially stitching all four corneal maps into a single 224×224 canvas and processing it through a pretrained Vision Transformer backbone, with a multilayer perceptron branch that encodes six structured clinical features. The fused representation passes through a dense clas

### Step 4: Perform AI Detection using a Hugging Face Model (Chunking Strategy)

As your document is quite long (around 30,000 words), it's crucial to understand that most Hugging Face models, including the `roberta-base-openai-detector`, have a maximum input length (e.g., 512 tokens). Directly feeding the entire document would result in truncation, meaning only the beginning would be analyzed.

To overcome this, we will implement a **chunking strategy**:

1.  **Split the document**: The `file_content` will be divided into smaller, overlapping chunks. The overlap helps maintain context across chunk boundaries.
2.  **Process each chunk**: Each chunk will be passed individually to the AI detection model.
3.  **Aggregate results**: We will collect the predictions (label and score) for each chunk. For a comprehensive overview, we'll summarize how many chunks were classified as 'human' versus 'AI', and provide the average confidence scores for each category. This will give you an idea of the overall AI-generated likelihood across the entire document.

Keep in mind that AI detection models are not perfect and can sometimes misclassify text. The confidence score indicates the model's certainty for each chunk.

In [8]:
from transformers import pipeline

# Load a pre-trained AI detection model
# A commonly used model for this purpose is 'roberta-base-openai-detector'
# This will download the model the first time it's run.
print("Loading AI detection model...")
detector = pipeline("text-classification", model="roberta-base-openai-detector")
print("Model loaded.")

# --- Chunking Strategy for Long Documents ---

# Define chunk size and overlap (in characters).
# A token is roughly 4 characters, so 512 tokens is approx 2048 characters.
# We'll use a slightly smaller chunk size to be safe and add overlap.
CHUNK_SIZE = 1500  # characters
OVERLAP = 200    # characters

# Split the document into chunks
chunks = []
start = 0
while start < len(file_content):
    end = start + CHUNK_SIZE
    chunk = file_content[start:end]
    chunks.append(chunk)
    start += (CHUNK_SIZE - OVERLAP)

print(f"\nDocument split into {len(chunks)} chunks for analysis.")

all_results = []

print("Performing AI detection on each chunk...")
for i, chunk in enumerate(chunks):
    try:
        # The detector returns a list of dictionaries, usually one for the input text.
        chunk_result = detector(chunk)
        if chunk_result:
            all_results.append(chunk_result[0])
        # Optional: Print progress for long documents
        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(chunks)} chunks...")
    except Exception as e:
        print(f"Error processing chunk {i}: {e}")
        all_results.append({'label': 'ERROR', 'score': 0.0})

print("\n--- AI Detection Results Summary ---")

human_count = 0
ai_count = 0
human_scores = []
ai_scores = []
error_count = 0

for result in all_results:
    label = result['label']
    score = result['score']
    if label == 'Real': # The model outputs 'Real' for human-generated content
        human_count += 1
        human_scores.append(score)
    elif label == 'Fake': # The model outputs 'Fake' for AI-generated content
        ai_count += 1
        ai_scores.append(score)
    else:
        # This handles unexpected labels or cases where the model couldn't classify
        error_count += 1

total_chunks = len(all_results)

print(f"Total Chunks Analyzed: {total_chunks}")
print(f"Chunks Classified as Human (Real): {human_count} ({(human_count/total_chunks)*100:.2f}%) - Average Confidence: {sum(human_scores)/len(human_scores):.4f}" if human_count > 0 else "No chunks classified as Human (Real).")
print(f"Chunks Classified as AI (Fake): {ai_count} ({(ai_count/total_chunks)*100:.2f}%) - Average Confidence: {sum(ai_scores)/len(ai_scores):.4f}" if ai_count > 0 else "No chunks classified as AI (Fake).")
if error_count > 0:
    print(f"Chunks with Unhandled Labels or Errors: {error_count}")

print("\nInterpretation:")
print("- 'Fake' typically means the model predicts AI-generated content.")
print("- 'Real' typically means the model predicts human-generated content.")
print("- The scores indicate the confidence of the prediction for each chunk.")
print("A higher percentage of 'Fake' chunks suggests the document might contain significant AI-generated content.")

Loading AI detection model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.

Document split into 27 chunks for analysis.
Performing AI detection on each chunk...
Processed 10/27 chunks...
Processed 20/27 chunks...

--- AI Detection Results Summary ---
Total Chunks Analyzed: 27
Chunks Classified as Human (Real): 27 (100.00%) - Average Confidence: 0.9992
No chunks classified as AI (Fake).

Interpretation:
- 'Fake' typically means the model predicts AI-generated content.
- 'Real' typically means the model predicts human-generated content.
- The scores indicate the confidence of the prediction for each chunk.
A higher percentage of 'Fake' chunks suggests the document might contain significant AI-generated content.


### Step 5: Perform AI Detection using a Different Hugging Face Model (ChatGPT Detector)

Now, let's try a different AI detection model: `Hello-SimpleAI/chatgpt-detector-roberta`. This model is specifically fine-tuned to detect text generated by ChatGPT. The labels for this model are typically 'Human' and 'ChatGPT'. We will continue to use the chunking strategy to process the entire document.

In [9]:
from transformers import pipeline

# Load a different pre-trained AI detection model: chatgpt-detector-roberta
print("Loading chatgpt-detector-roberta model...")
detector_chatgpt = pipeline("text-classification", model="Hello-SimpleAI/chatgpt-detector-roberta")
print("Model loaded.")

# Reuse the same chunking parameters
# CHUNK_SIZE = 1500  # characters
# OVERLAP = 200    # characters

# Reuse the chunks generated previously, or regenerate if needed
# For this example, we assume `chunks` is still available from the previous run.
# If `chunks` is not defined, you would uncomment and re-run the chunking logic here.
# For safety and re-runnability, let's include the chunking logic again.
chunks = []
start = 0
while start < len(file_content):
    end = start + CHUNK_SIZE
    chunk = file_content[start:end]
    chunks.append(chunk)
    start += (CHUNK_SIZE - OVERLAP)

print(f"\nDocument split into {len(chunks)} chunks for analysis (using the same chunking parameters).")

all_results_chatgpt = []

print("Performing AI detection on each chunk with chatgpt-detector...")
for i, chunk in enumerate(chunks):
    try:
        chunk_result = detector_chatgpt(chunk)
        if chunk_result:
            all_results_chatgpt.append(chunk_result[0])
        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(chunks)} chunks...")
    except Exception as e:
        print(f"Error processing chunk {i}: {e}")
        all_results_chatgpt.append({'label': 'ERROR', 'score': 0.0})

print("\n--- AI Detection Results Summary (chatgpt-detector-roberta) ---")

human_count_chatgpt = 0
ai_count_chatgpt = 0
human_scores_chatgpt = []
ai_scores_chatgpt = []
error_count_chatgpt = 0

for result in all_results_chatgpt:
    label = result['label']
    score = result['score']
    if label == 'Human':
        human_count_chatgpt += 1
        human_scores_chatgpt.append(score)
    elif label == 'ChatGPT':
        ai_count_chatgpt += 1
        ai_scores_chatgpt.append(score)
    else:
        error_count_chatgpt += 1

total_chunks_chatgpt = len(all_results_chatgpt)

print(f"Total Chunks Analyzed: {total_chunks_chatgpt}")
print(f"Chunks Classified as Human: {human_count_chatgpt} ({(human_count_chatgpt/total_chunks_chatgpt)*100:.2f}%) - Average Confidence: {sum(human_scores_chatgpt)/len(human_scores_chatgpt):.4f}" if human_count_chatgpt > 0 else "No chunks classified as Human.")
print(f"Chunks Classified as ChatGPT: {ai_count_chatgpt} ({(ai_count_chatgpt/total_chunks_chatgpt)*100:.2f}%) - Average Confidence: {sum(ai_scores_chatgpt)/len(ai_scores_chatgpt):.4f}" if ai_count_chatgpt > 0 else "No chunks classified as ChatGPT.")
if error_count_chatgpt > 0:
    print(f"Chunks with Unhandled Labels or Errors: {error_count_chatgpt}")

print("\nInterpretation:")
print("- 'ChatGPT' means the model predicts text generated by ChatGPT.")
print("- 'Human' means the model predicts human-generated content.")
print("- The scores indicate the confidence of the prediction for each chunk.")
print("A higher percentage of 'ChatGPT' chunks suggests the document might contain significant AI-generated content (specifically from ChatGPT or similar models).")

Loading chatgpt-detector-roberta model...


config.json:   0%|          | 0.00/858 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/391 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Model loaded.

Document split into 27 chunks for analysis (using the same chunking parameters).
Performing AI detection on each chunk with chatgpt-detector...
Processed 10/27 chunks...
Processed 20/27 chunks...

--- AI Detection Results Summary (chatgpt-detector-roberta) ---
Total Chunks Analyzed: 27
Chunks Classified as Human: 27 (100.00%) - Average Confidence: 0.9995
No chunks classified as ChatGPT.

Interpretation:
- 'ChatGPT' means the model predicts text generated by ChatGPT.
- 'Human' means the model predicts human-generated content.
- The scores indicate the confidence of the prediction for each chunk.
A higher percentage of 'ChatGPT' chunks suggests the document might contain significant AI-generated content (specifically from ChatGPT or similar models).
